In [15]:
# import the necessary packages
import re
import warnings
import pandas as pd
from tf.app import use

warnings.filterwarnings("ignore")

In [16]:
A = use('etcbc/bhsa', hoist=globals())

This is Text-Fabric 9.2.5
Api reference : https://annotation.github.io/text-fabric/tf/cheatsheet.html

122 features found and 0 ignored


In [17]:
verses = ["{} {}:{}".format(*T.sectionFromNode(verse)) for verse in F.otype.s('verse')]

In [18]:
texts = [" ".join([T.text(word, fmt='text-orig-full').strip() for word in L.d(verse, otype="word")]) for verse in F.otype.s('verse')]

In [19]:
df = pd.DataFrame({'Verse' : verses, 'Text' : texts})

df.head()

,Verse,Text
0,Genesis 1:1,בְּ רֵאשִׁ֖ית בָּרָ֣א אֱלֹהִ֑ים אֵ֥ת הַ שָּׁמַ...
1,Genesis 1:2,וְ הָ אָ֗רֶץ הָיְתָ֥ה תֹ֨הוּ֙ וָ בֹ֔הוּ וְ חֹ֖...
2,Genesis 1:3,וַ יֹּ֥אמֶר אֱלֹהִ֖ים יְהִ֣י אֹ֑ור וַֽ יְהִי־ ...
3,Genesis 1:4,וַ יַּ֧רְא אֱלֹהִ֛ים אֶת־ הָ אֹ֖ור כִּי־ טֹ֑וב...
4,Genesis 1:5,וַ יִּקְרָ֨א אֱלֹהִ֤ים׀ לָ אֹור֙ יֹ֔ום וְ לַ ...


In [20]:
hex_codes = ["\\u" + str(hex(x).replace("x", "")) for x in range(0x5b0, 0x5bd)]

In [21]:
hex_codes.extend(["\\u" + str(hex(x).replace("x", "")) for x in range (0x5d0, 0x5eb)])

In [22]:
hex_codes.extend(["\\u05c7", "\\u05be"]) #tried using "\\u05bf" for the maqqef, but that didn't work see below

In [23]:
patterns = re.compile("[^\s|" + "|".join(hex_codes) + "]")

In [24]:
df['Text'] = df['Text'].str.replace(patterns, "")

df.head()

,Verse,Text
0,Genesis 1:1,בְּ רֵאשִית בָּרָא אֱלֹהִים אֵת הַ שָּמַיִם וְ...
1,Genesis 1:2,וְ הָ אָרֶץ הָיְתָה תֹהוּ וָ בֹהוּ וְ חֹשֶךְ ע...
2,Genesis 1:3,וַ יֹּאמֶר אֱלֹהִים יְהִי אֹור וַ יְהִי־ אֹור
3,Genesis 1:4,וַ יַּרְא אֱלֹהִים אֶת־ הָ אֹור כִּי־ טֹוב וַ ...
4,Genesis 1:5,וַ יִּקְרָא אֱלֹהִים לָ אֹור יֹום וְ לַ חֹשֶ...


In [25]:
df['Text'] = df['Text'].str.replace("\s\\u05e1$", "")
df['Text'] = df['Text'].str.replace("\s\\u05e4$", "")
df['Text'] = df['Text'].str.replace('־', "") #tried using a hexcode for maqqef, but that didn't work

In [26]:
replacement_vals_books = {'Genesis' : 'Gen', 
                          'Exodus' : 'Exod', 
                          'Numbers' : 'Num', 
                          'Leviticus' : 'Lev', 
                          'Deuteronomy' : 'Deut', 
                          'Joshua' : 'Josh', 
                          'Judges' : 'Jud',
                          '1_Samuel' : '1 Sam', 
                          '2_Samuel' : '2 Sam', 
                          '1_Kings' : '1 Kgs', 
                          '2_Kings' : '2 Kgs', 
                          'Isaiah' : 'Isa', 
                          'Jeremiah' : 'Jer', 
                          'Ezekiel' : 'Ezek', 
                          'Hosea' : 'Hos', 
                          'Obadiah' : 'Obad', 
                          'Michah' : 'Mic', 
                          'Nahum' : 'Nah', 
                          'Habakkuk' : 'Hab', 
                          'Zephaniah' : 'Zeph', 
                          'Haggai' : 'Hag', 
                          'Zechariah' : 'Zech', 
                          'Malachi' : 'Mal', 
                          'Psalms' : 'Ps', 
                          'Proverbs' : 'Prov', 
                          'Song of Songs' : 'Song', 
                          'Ecclesiastes' : 'Eccl', 
                          'Lamentations' : 'Lam', 
                          'Esther' : 'Esth', 
                          'Daniel' : 'Dan', 
                          'Nehemiah' : 'Neh', 
                          '1_Chronicles' : '1 Chr', 
                          '2_Chronicles' : '2 Chr'}

df['Verse'].replace(replacement_vals_books, inplace=True, regex=True)

In [28]:
df.set_index('Verse', inplace=True)

In [29]:
df.drop('Jer 10:11', inplace=True)
df.drop(df.loc['Dan 2:5' : 'Dan 7:28'].index, inplace=True)
df.drop(df.loc['Ezra 4:8' : 'Ezra 6:18'].index, inplace=True)
df.drop(df.loc['Ezra 7:12' : 'Ezra 7:26'].index, inplace=True)

In [30]:
df.loc['Dan 2:4']['Text'] = 'וַ יְדַבְּרוּ הַ כַּשְדִּים לַ  מֶּלֶךְ אֲרָמִית'

In [31]:
df['Text'] = df['Text'].str.normalize('NFC')

In [32]:
df.to_csv('BERiT_training_data.csv')